# Tools and Routing

In [ ]:
# 读取 .env 里的 OPENAI_API_KEY
import os
import openai

from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [ ]:
# 【版本兼容修复】langchain.agents 已不再导出 tool 装饰器，
# @tool 这个把普通函数变成 LangChain "工具"（Tool）的装饰器现在在 langchain_core.tools 下
from langchain_core.tools import tool

In [ ]:
# @tool 装饰器会自动把这个函数包装成一个 LangChain Tool 对象：
# - 函数名变成 tool.name
# - docstring 变成 tool.description（模型据此判断何时调用这个工具）
# - 类型注解自动推导出 tool.args 的 JSON Schema
@tool
def search(query:str)->str:
    """Search for weather online"""
    return "42f"

In [ ]:
search.name

In [ ]:
search.description

In [ ]:
# args 是从函数签名自动推导出的参数 JSON Schema
search.args

In [ ]:
# 如果想自定义参数的 description（而不是只靠类型注解），可以单独定义一个 pydantic 模型作为 args_schema
from pydantic import BaseModel,Field
class SearchInput(BaseModel):
    query:str=Field(description="Thing to search for")

In [ ]:
# 注：这里重新定义 search，但还没有真正把 SearchInput 接进来（args_schema 没传），
# 后面的 get_current_temperature 例子会展示完整的 @tool(args_schema=...) 用法
@tool
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

In [ ]:
search.args

In [ ]:
# .run() 是 Tool 对象真正"执行"的方法（Tool 对象本身在新版 langchain_core 里已经不能直接 search(...) 调用了）
search.run("sf")

In [ ]:
import requests
from pydantic import BaseModel, Field
import datetime

# 用 pydantic 模型明确定义输入参数的 schema（包含 description），这样转成 OpenAI function 时描述更完整
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

# args_schema=OpenMeteoInput：显式指定参数 schema（而不是仅从函数签名自动推导），
# 这样 description 等元信息会被保留下来
@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request（Open-Meteo 是免费公开 API，不需要 API key）
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']

    # 从返回的逐小时数据里，找到离当前 UTC 时间最近的那个时间点对应的温度
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f'The current temperature is {current_temperature}°C'

In [ ]:
get_current_temperature.name

In [ ]:
get_current_temperature.description

In [ ]:
get_current_temperature.args

In [ ]:
# 【版本兼容修复】langchain.tools.render 已不存在，format_tool_to_openai_function 现在在 langchain_community.tools.render 下
# （这个函数是旧版 function calling 专用的转换器；新代码更推荐用 tools/tool_choice，但这里保持课程原本写法）
# 作用：把一个 LangChain Tool 对象转换成 OpenAI function calling 需要的 JSON Schema
from langchain_community.tools.render import format_tool_to_openai_function

In [ ]:
format_tool_to_openai_function(get_current_temperature)

In [ ]:
# 【版本兼容修复 / 真实 bug】新版 langchain_core 里 StructuredTool 不再支持像普通函数一样直接 __call__，
# 直接 get_current_temperature({...}) 会报 "TypeError: 'StructuredTool' object is not callable"，
# 必须改用 .invoke(...) 来真正执行这个工具
get_current_temperature.invoke({"latitude": 13, "longitude": 14})

In [ ]:
import wikipedia
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            # 【真实 bug 修复】原代码写的是 self.wiki_client.exceptions.PageError /
            # self.wiki_client.exceptions.DisambiguationError。
            # 但这只是一个普通函数，既没有 self，也没有 wiki_client 这个东西
            # （这是从某个类方法里复制粘贴过来时忘记改的残留代码），运行时会直接 NameError: name 'self' is not defined。
            # 正确写法应该直接用已经 import 的 wikipedia 模块本身的 exceptions
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [ ]:
search_wikipedia.name

In [ ]:
search_wikipedia.description

In [ ]:
format_tool_to_openai_function(search_wikipedia)

In [ ]:
# 【版本兼容修复】同上，Tool 对象不能直接调用，改用 .invoke(...)
# 这一步会真正发起 Wikipedia 搜索请求（普通 HTTP 请求，不需要 OpenAI key）
search_wikipedia.invoke({"query": "langchain"})

In [ ]:
# 【版本兼容修复】langchain.chains.openai_functions.openapi / langchain.utilities.openapi 已不存在。
# openapi_spec_to_openai_fn 这个函数在 langchain 1.x 里被归类为"旧版/经典"用法，
# 搬到了单独的 langchain_classic 包（pip 安装 langchain 1.x 时会自动带上这个依赖包）。
# OpenAPISpec 仍留在 langchain_community.utilities.openapi 下。
#
# 【环境限制，未擅自安装新依赖】OpenAPISpec 内部依赖可选包 openapi_pydantic 来解析 OpenAPI 规范；
# 当前 venv 没有安装 openapi_pydantic（不在题目给出的已装包清单里），
# 这会导致 OpenAPISpec.from_text(...) 在下面几个 cell 里抛出
# AttributeError: 'super' object has no attribute 'parse_obj'（不是网络问题，是纯本地依赖缺失）。
# 如果需要跑通这一节，需要额外执行 `pip install openapi-pydantic`（未经你确认，这里没有擅自安装）。
from langchain_classic.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain_community.utilities.openapi import OpenAPISpec

In [ ]:
# 一个精简版的 Swagger Petstore OpenAPI 规范（JSON 字符串），演示如何把 REST API 规范
# 自动转换成 OpenAI function calling 的函数列表
text = """
{
  "openapi": "3.0.0",
  "info": {
    "version": "1.0.0",
    "title": "Swagger Petstore",
    "license": {
      "name": "MIT"
    }
  },
  "servers": [
    {
      "url": "http://petstore.swagger.io/v1"
    }
  ],
  "paths": {
    "/pets": {
      "get": {
        "summary": "List all pets",
        "operationId": "listPets",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "limit",
            "in": "query",
            "description": "How many items to return at one time (max 100)",
            "required": false,
            "schema": {
              "type": "integer",
              "maximum": 100,
              "format": "int32"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "A paged array of pets",
            "headers": {
              "x-next": {
                "description": "A link to the next page of responses",
                "schema": {
                  "type": "string"
                }
              }
            },
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pets"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      },
      "post": {
        "summary": "Create a pet",
        "operationId": "createPets",
        "tags": [
          "pets"
        ],
        "responses": {
          "201": {
            "description": "Null response"
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    },
    "/pets/{petId}": {
      "get": {
        "summary": "Info for a specific pet",
        "operationId": "showPetById",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "petId",
            "in": "path",
            "required": true,
            "description": "The id of the pet to retrieve",
            "schema": {
              "type": "string"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "Expected response to a valid request",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pet"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "Pet": {
        "type": "object",
        "required": [
          "id",
          "name"
        ],
        "properties": {
          "id": {
            "type": "integer",
            "format": "int64"
          },
          "name": {
            "type": "string"
          },
          "tag": {
            "type": "string"
          }
        }
      },
      "Pets": {
        "type": "array",
        "maxItems": 100,
        "items": {
          "$ref": "#/components/schemas/Pet"
        }
      },
      "Error": {
        "type": "object",
        "required": [
          "code",
          "message"
        ],
        "properties": {
          "code": {
            "type": "integer",
            "format": "int32"
          },
          "message": {
            "type": "string"
          }
        }
      }
    }
  }
}
"""

In [ ]:
# 解析 OpenAPI 规范字符串（如果本机没装 openapi_pydantic，这里会报 AttributeError，见上面 cell 的说明）
spec = OpenAPISpec.from_text(text)

In [ ]:
# 把 OpenAPI spec 里每个 endpoint 转换成一个 OpenAI function schema（pet_openai_functions）
# 以及对应的可调用函数（pet_callables，真正发起 HTTP 请求去调用 petstore API）
pet_openai_functions, pet_callables = openapi_spec_to_openai_fn(spec)

In [ ]:
pet_openai_functions

In [ ]:
# 【版本兼容修复】langchain.chat_models 已不再导出 ChatOpenAI，搬到了 langchain_openai
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI(temperature=0).bind(functions=pet_openai_functions)

In [ ]:
# 期望：模型选择调用 listPets 这个函数
model.invoke("what are three pets names")

In [ ]:
# 期望：模型选择调用 showPetById，并提取出 petId=42
model.invoke("tell me about pet with id 42")

In [ ]:
# 切回真正会用到的两个工具：查天气 + 查维基百科，把它们都转换成 OpenAI function schema 并绑定到模型
functions = [
    format_tool_to_openai_function(f) for f in [
        search_wikipedia, get_current_temperature
    ]
]
model = ChatOpenAI(temperature=0).bind(functions=functions)

In [ ]:
# 期望：模型选择调用 get_current_temperature
model.invoke("what is the weather in sf right now")

In [ ]:
# 期望：模型选择调用 search_wikipedia
model.invoke("what is langchain")

In [ ]:
# 【版本兼容修复】langchain.prompts 已不存在，ChatPromptTemplate 搬到了 langchain_core.prompts
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])
chain = prompt | model

In [ ]:
chain.invoke({"input": "what is the weather in sf right now"})

In [ ]:
# 【版本兼容修复】langchain.agents.output_parsers 已不存在，
# OpenAIFunctionsAgentOutputParser 现在在 langchain_classic.agents.output_parsers 下（旧版 agent 相关代码归到了 langchain_classic）
# 作用：把模型输出解析成 AgentAction（要调用哪个工具）或 AgentFinish（直接给出最终答案）
from langchain_classic.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [ ]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [ ]:
result = chain.invoke({"input": "what is the weather in sf right now"})

In [ ]:
# 当模型决定调用工具时，result 的类型是 AgentActionMessageLog（AgentAction 的子类）
type(result)

In [ ]:
# result.tool 是模型决定要调用的工具名字（字符串）
result.tool

In [ ]:
# result.tool_input 是模型生成的调用参数（dict）
result.tool_input

In [ ]:
# 【版本兼容修复 / 真实 bug】同前面一样，Tool 对象不能直接调用，要用 .invoke(...)
get_current_temperature.invoke(result.tool_input)

In [ ]:
# 跟任何工具都无关的输入，期望模型直接给出最终回答而不是调用工具
result = chain.invoke({"input": "hi!"})

In [ ]:
# 这次 result 的类型是 AgentFinish，表示模型不打算调用任何工具，直接给出了最终答案
type(result)

In [ ]:
# AgentFinish 的最终回答存在 return_values['output'] 里
result.return_values

In [ ]:
# 【版本兼容修复】langchain.schema.agent 已不存在，AgentFinish 现在在 langchain_core.agents 下
from langchain_core.agents import AgentFinish

# route：根据链的输出类型做分发——
# - 如果是 AgentFinish，说明模型已经给出最终答案，直接返回文字
# - 否则说明模型要调用某个工具，根据 result.tool 找到对应的 Tool 对象，用 .run() 真正执行它
def route(result):
    if isinstance(result, AgentFinish):
        return result.return_values['output']
    else:
        tools = {
            "search_wikipedia": search_wikipedia,
            "get_current_temperature": get_current_temperature,
        }
        return tools[result.tool].run(result.tool_input)

In [ ]:
# 完整流水线：prompt -> model -> 解析成 AgentAction/AgentFinish -> route 自动分发执行
chain = prompt | model | OpenAIFunctionsAgentOutputParser() | route

In [ ]:
result = chain.invoke({"input": "What is the weather in san francisco right now?"})

In [ ]:
result

In [ ]:
result = chain.invoke({"input": "What is langchain?"})

In [ ]:
result

In [ ]:
# 期望：route 直接返回 AgentFinish 里的文字答案，不调用任何工具
chain.invoke({"input": "hi!"})